# 03 — Entraîner et évaluer le modèle de classification

Exécuter les cellules dans l’ordre, de haut en bas. Le notebook vérifie les images de `data/raw`, crée un split stratifié Train/Validation/Test (70/15/15), augmente et équilibre uniquement Train, entraîne InceptionV3 en deux phases, puis évalue une seule fois sur Test. Les données sources ne sont pas modifiées.

**Pré-requis :** dépendances du projet installées et accès Internet au premier lancement pour télécharger les poids ImageNet d’InceptionV3. Sur Vast.ai, sélectionner le kernel Python de l’environnement avec TensorFlow et GPU.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError(
        f"Racine du projet introuvable depuis {Path.cwd()}. Ouvrir le notebook depuis le projet."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing import (
    creer_datasets_images,
    inspecter_dataset_images,
    repartir_images,
)
from src.model_architecture import build_inception_model, prepare_for_fine_tuning

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS_FEATURE_EXTRACTION = 10
EPOCHS_FINE_TUNING = 10
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("TensorFlow :", tf.__version__)
print("GPU(s) disponibles :", tf.config.list_physical_devices("GPU"))
print("Dossier du projet :", PROJECT_ROOT)

In [ ]:
images, fichiers_invalides = inspecter_dataset_images(PROJECT_ROOT / "data" / "raw")
if images.empty:
    raise ValueError("Aucune image valide trouvée dans data/raw.")

print(f"Images valides : {len(images)}")
print(f"Fichiers invalides/ignorés : {len(fichiers_invalides)}")
display(images.groupby("label").size().rename("images valides").to_frame())
if not fichiers_invalides.empty:
    display(fichiers_invalides.head(10))

repartitions = repartir_images(
    images, taille_test=0.15, taille_validation=0.15, seed=SEED
)
effectifs = {split: len(table) for split, table in repartitions.items()}
print("Effectifs avant équilibrage :", effectifs)
for split, table in repartitions.items():
    print(f"\n{split.capitalize()} par classe :")
    display(table.groupby("label").size().rename("images").to_frame())

datasets, split_workspace, class_names, effectifs_classes = creer_datasets_images(
    repartitions,
    taille_image=IMAGE_SIZE,
    taille_lot=BATCH_SIZE,
    seed=SEED,
    augmenter=True,
    equilibrer=True,
)
print("Ordre des classes/indices :", class_names)
print("Augmentation et équilibrage appliqués uniquement à Train.")

class_names_path = MODELS_DIR / "class_names.json"
class_names_path.write_text(
    json.dumps(class_names, ensure_ascii=False, indent=2), encoding="utf-8"
)
with open(class_names_path, encoding="utf-8") as fichier:
    print("Correspondance sauvegardée :", json.load(fichier))

## Phase 1 — Extraction des caractéristiques

La base InceptionV3 pré-entraînée est figée; seule la tête de classification apprend. Le meilleur checkpoint est sélectionné avec la précision de Validation.

In [ ]:
model, base_model = build_inception_model(
    num_classes=len(class_names), input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)
)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

best_model_path = MODELS_DIR / "best_model.keras"
callbacks_feature_extraction = [
    ModelCheckpoint(
        best_model_path,
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    EarlyStopping(
        monitor="val_loss", patience=4, restore_best_weights=True, verbose=1
    ),
]

history_feature_extraction = model.fit(
    datasets["train"],
    validation_data=datasets["validation"],
    epochs=EPOCHS_FEATURE_EXTRACTION,
    callbacks=callbacks_feature_extraction,
)

## Phase 2 — Fine-tuning

On dégèle les dernières couches InceptionV3 avec un taux d’apprentissage plus faible. Les couches BatchNormalization restent figées pour stabiliser l’apprentissage.

In [ ]:
model = prepare_for_fine_tuning(
    model, base_model, unfreeze_from_layer=249
)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks_fine_tuning = [
    ModelCheckpoint(
        best_model_path,
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    EarlyStopping(
        monitor="val_loss", patience=4, restore_best_weights=True, verbose=1
    ),
]
history_fine_tuning = model.fit(
    datasets["train"],
    validation_data=datasets["validation"],
    epochs=EPOCHS_FINE_TUNING,
    callbacks=callbacks_fine_tuning,
)

final_model_path = MODELS_DIR / "final_model.keras"
model.save(final_model_path)
print("Modèle sauvegardé :", final_model_path)

## Évaluation finale sur Test

Le jeu Test n’a pas servi à l’entraînement ni au choix des poids. Cette cellule affiche les métriques, le rapport par classe, la matrice de confusion et les courbes d’apprentissage, puis sauvegarde les résultats dans `models/`.

In [ ]:
test_metrics = model.evaluate(datasets["test"], verbose=1, return_dict=True)
print("Métriques Test :")
for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}")

y_true = np.concatenate([labels.numpy() for _, labels in datasets["test"]])
y_probabilities = model.predict(datasets["test"], verbose=1)
y_pred = np.argmax(y_probabilities, axis=1)
report = classification_report(
    y_true,
    y_pred,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)
display(pd.DataFrame(report).transpose().round(4))

cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))
fig_confusion, ax_confusion = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    ax=ax_confusion,
)
ax_confusion.set_title("Matrice de confusion — Test")
ax_confusion.set_xlabel("Prédit")
ax_confusion.set_ylabel("Réel")

phase1 = history_feature_extraction.history
phase2 = history_fine_tuning.history
train_accuracy = phase1.get("accuracy", []) + phase2.get("accuracy", [])
val_accuracy = phase1.get("val_accuracy", []) + phase2.get("val_accuracy", [])
train_loss = phase1.get("loss", []) + phase2.get("loss", [])
val_loss = phase1.get("val_loss", []) + phase2.get("val_loss", [])
epochs = np.arange(1, len(train_accuracy) + 1)
fig_curves, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(epochs, train_accuracy, label="Train")
axes[0].plot(epochs, val_accuracy, label="Validation")
axes[0].axvline(len(phase1.get("accuracy", [])) + 0.5, color="gray", linestyle="--")
axes[0].set_title("Accuracy par epoch")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].plot(epochs, train_loss, label="Train")
axes[1].plot(epochs, val_loss, label="Validation")
axes[1].axvline(len(phase1.get("loss", [])) + 0.5, color="gray", linestyle="--")
axes[1].set_title("Loss par epoch")
axes[1].set_xlabel("Epoch")
axes[1].legend()

confusion_path = MODELS_DIR / "confusion_matrix_test.png"
curves_path = MODELS_DIR / "training_curves.png"
fig_confusion.savefig(confusion_path, dpi=150, bbox_inches="tight")
fig_curves.tight_layout()
fig_curves.savefig(curves_path, dpi=150, bbox_inches="tight")
plt.show()

metrics_path = MODELS_DIR / "test_metrics.json"
metrics_path.write_text(
    json.dumps(
        {"metrics": {key: float(value) for key, value in test_metrics.items()}, "classification_report": report},
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
print("Résultats sauvegardés :", metrics_path, confusion_path, curves_path)

## Nettoyage après la fin

Exécuter cette dernière cellule seulement après le training et l’évaluation : elle supprime l’espace temporaire qui contient les liens/copies de travail, pas les images originales.

In [ ]:
split_workspace.cleanup()
print("Espace temporaire libéré. Les images originales sont intactes.")